# Theorem 21 — support projector perturbation

**Formal source:** [`../21_support_projector_perturbation_bound.md`](../21_support_projector_perturbation_bound.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
gramian = np.diag([3.0, 1.0])
perturbation = np.array([[0, 0.08], [0.08, 0]])

def top_projector(matrix):
    vector = np.linalg.eigh(matrix)[1][:, [-1]]
    return vector @ vector.T

error = np.linalg.norm(top_projector(gramian + perturbation) - top_projector(gramian), 2)
bound = 2 * np.linalg.norm(perturbation, 2) / 2
assert error <= bound + 1e-12
before = int(np.sum(np.linalg.eigvalsh(np.diag([2.01, 1])) >= 2))
after = int(np.sum(np.linalg.eigvalsh(np.diag([1.99, 1])) >= 2))
assert before != after
print({"projector_error": float(error), "bound": float(bound), "near_threshold_rank": [before, after]})

In [ ]:
print('THEORY_DEMO_PASS::21_support_projector_perturbation_bound')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')